In [1]:
import pandas as pd
import numpy as np

# Cargamos la base limpia que generamos en el pipeline anterior
df = pd.read_excel("../data/final_data/df_base_limpia.xlsx")

df_base = pd.read_excel("../data/final_data/df_base_con_flags.xlsx")

# Cargamos el diccionario de hospitales
hospitales = pd.read_csv("../data/hospitales_coordenadas.csv")

print(f"Base a auditar: {len(df)} episodios válidos.")
print(f"Base limpia y no filtrada: {len(df_base)} episodios válidos.")


Base a auditar: 21790 episodios válidos.
Base limpia y no filtrada: 29697 episodios válidos.


### Universo 1: pacientes con solo 1 registro

In [2]:
# UNIVERSO 1 - Pacientes con 1 solo registro
# reporte de errrores SOLO PARA CASO TRIVIAL (Pacientes con 1 solo registro)
############### borrar despues
print("\n" + "="*65)
print("📊 REPORTE DE IMPACTO: CASO TRIVIAL (Pacientes con 1 solo registro)")
print("="*65)

# 1. Identificamos los pacientes que tienen exactamente 1 registro en toda la base
conteo_pacientes = df_base['paciente_id'].value_counts()
ids_triviales = conteo_pacientes[conteo_pacientes == 1].index

# 2. Filtramos la base para quedarnos solo con este universo
df_trivial = df_base[df_base['paciente_id'].isin(ids_triviales)].copy()
total_trivial = len(df_trivial)

print(f"Total de episodios en el universo trivial: {total_trivial} filas\n")

# 3. Calculamos las anomalías sobre este subconjunto
filtros_trivial = [
    ("1. Sin ID paciente", df_trivial['paciente_id'].isna().sum()), 
    ("2. Sin Hospital", (df_trivial['hospital_origen'].isna() | (df_trivial['hospital_origen'] == '')).sum()),
    ("3. Sin Fecha Ingreso", df_trivial['fecha_ingreso'].isna().sum()),
    ("4. Sin Fecha Egreso", df_trivial['fecha_egreso'].isna().sum()),
    ("5. Fechas Incoherentes (< 0)", df_trivial['flag_fechas_incoherentes'].sum()),
    ("6. Edades Raras", df_trivial['flag_edad_rara'].sum()),
    ("7. Estancias Largas (> 60)", df_trivial['flag_duracion_larga'].sum()),
    
    # OJO ACÁ: Ahora sí el .sum() está envolviendo solo a las dos condiciones lógicas
    ("8. Err. Admin / Traslado Huérfano", ((df_trivial['flag_egreso_admin_invalido']) | (df_trivial['tipo_egreso'] == 'traslado')).sum())
]

for nombre, cantidad in filtros_trivial:
    pct = (cantidad / total_trivial) * 100 if total_trivial > 0 else 0
    print(f"{nombre:<35}: {cantidad:>6} filas | {pct:>6.2f}%")
print("="*65 + "\n")


# Calculamos cuántos episodios triviales tienen AL MENOS UN ERROR
mask_errores_trivial = (
    (df_trivial['paciente_id'].isna()) |
    (df_trivial['hospital_origen'].isna()) | (df_trivial['hospital_origen'] == '') |
    (df_trivial['fecha_ingreso'].isna()) |
    (df_trivial['fecha_egreso'].isna()) |
    (df_trivial['flag_fechas_incoherentes']) |
    (df_trivial['flag_edad_rara']) |
    (df_trivial['flag_duracion_larga']) |
    (df_trivial['flag_egreso_admin_invalido']) | 
    (df_trivial['tipo_egreso'] == 'traslado')
)

triviales_con_error = mask_errores_trivial.sum()
triviales_perfectos = total_trivial - triviales_con_error

pct_error = (triviales_con_error / total_trivial) * 100 if total_trivial > 0 else 0
pct_perfectos = (triviales_perfectos / total_trivial) * 100 if total_trivial > 0 else 0

print("RESUMEN DE CALIDAD - CASO TRIVIAL (1 Solo Registro):")
print(f"✅ Casos Perfectos (Listos para analizar) : {triviales_perfectos:>6} filas | {pct_perfectos:>6.2f}%")
print(f"❌ Casos con Errores (A filtrar/auditar)  : {triviales_con_error:>6} filas | {pct_error:>6.2f}%")
print("="*65 + "\n")
#


# ignorando los errores 6, 7, 8
# 5. Calculamos cuántos episodios triviales tienen errores ESTRUCTURALES (Solo 1 al 5)
mask_errores_estructurales = (
    (df_trivial['paciente_id'].isna()) |
    (df_trivial['hospital_origen'].isna()) | (df_trivial['hospital_origen'] == '') |
    (df_trivial['fecha_ingreso'].isna()) |
    (df_trivial['fecha_egreso'].isna()) |
    (df_trivial['flag_fechas_incoherentes'])
)

triviales_con_error_est = mask_errores_estructurales.sum()
triviales_sanos_est = total_trivial - triviales_con_error_est

pct_error_est = (triviales_con_error_est / total_trivial) * 100 if total_trivial > 0 else 0
pct_sanos_est = (triviales_sanos_est / total_trivial) * 100 if total_trivial > 0 else 0

print("RESUMEN DE CALIDAD - ESTRUCTURAL (Ignorando 6, 7 y 8):")
print(f"✅ Casos Estructuralmente Sanos : {triviales_sanos_est:>6} filas | {pct_sanos_est:>6.2f}%")
print(f"❌ Casos con Errores Críticos   : {triviales_con_error_est:>6} filas | {pct_error_est:>6.2f}%")
print("="*65 + "\n")


📊 REPORTE DE IMPACTO: CASO TRIVIAL (Pacientes con 1 solo registro)
Total de episodios en el universo trivial: 25188 filas

1. Sin ID paciente                 :      0 filas |   0.00%
2. Sin Hospital                    :      0 filas |   0.00%
3. Sin Fecha Ingreso               :   3136 filas |  12.45%
4. Sin Fecha Egreso                :   3868 filas |  15.36%
5. Fechas Incoherentes (< 0)       :     33 filas |   0.13%
6. Edades Raras                    :    110 filas |   0.44%
7. Estancias Largas (> 60)         :    408 filas |   1.62%
8. Err. Admin / Traslado Huérfano  :   3865 filas |  15.34%

RESUMEN DE CALIDAD - CASO TRIVIAL (1 Solo Registro):
✅ Casos Perfectos (Listos para analizar) :  17590 filas |  69.83%
❌ Casos con Errores (A filtrar/auditar)  :   7598 filas |  30.17%

RESUMEN DE CALIDAD - ESTRUCTURAL (Ignorando 6, 7 y 8):
✅ Casos Estructuralmente Sanos :  20636 filas |  81.93%
❌ Casos con Errores Críticos   :   4552 filas |  18.07%



### Particion de universos

In [3]:
print("\n" + "="*70)
print("🌌 PARTICIÓN EN MACRO-UNIVERSOS (MECE ESTRICTO)")
print("="*70)

# 1. UNIVERSO 1: El Caso Trivial (Exactamente 1 registro)
conteo = df_base['paciente_id'].value_counts(dropna=False)
ids_u1 = conteo[conteo == 1].index
df_u1 = df_base[df_base['paciente_id'].isin(ids_u1)].copy()

# Preparamos los pacientes múltiples
df_multi = df_base[~df_base['paciente_id'].isin(ids_u1)].copy()
df_multi = df_multi.sort_values(['paciente_id', 'fecha_ingreso'])
df_multi['egreso_previo'] = df_multi.groupby('paciente_id')['fecha_egreso'].shift(1)

# Creamos funciones para validar cada historia clínica (grupo por paciente)
def evaluar_historial(g):
    # ¿Tiene algún día duplicado?
    tiene_mismo_dia = g.duplicated(subset=['fecha_ingreso']).any()
    
    # ¿Tiene algún solapamiento? (Ingreso actual < Egreso previo)
    tiene_solapamiento = (g['fecha_ingreso'] < g['egreso_previo']).any()
    
    # ¿Tiene fechas nulas? (Esto rompe las comparaciones)
    tiene_fechas_nulas = g['fecha_ingreso'].isna().any() or g['fecha_egreso'].isna().any()
    
    # DEFINICIÓN ESTRICTA DE U2 (Secuencia Limpia): 
    # No tiene mismo día, NO tiene fechas nulas, y TODOS sus ingresos (salvo el primero) son >= al egreso previo
    # Usamos [1:] para ignorar el primer registro del paciente, cuyo egreso_previo es NaT
    es_secuencia_limpia = (
        not tiene_mismo_dia and 
        not tiene_fechas_nulas and 
        (g['fecha_ingreso'].iloc[1:] >= g['egreso_previo'].iloc[1:]).all()
    )
    
    return pd.Series({
        'U3_MismoDia': tiene_mismo_dia,
        'U4_Solapamiento': (tiene_solapamiento and not tiene_mismo_dia), # Jerarquía
        'U2_Limpia': es_secuencia_limpia,
        'U5_Nulos_o_Raros': (not tiene_mismo_dia and not tiene_solapamiento and not es_secuencia_limpia)
    })

# Aplicamos la evaluación estricta paciente por paciente
resumen_pacientes = df_multi.groupby('paciente_id').apply(evaluar_historial)

# Asignamos los IDs a cada bolsa basándonos en sus reglas positivas
ids_u3 = resumen_pacientes[resumen_pacientes['U3_MismoDia']].index
ids_u4 = resumen_pacientes[resumen_pacientes['U4_Solapamiento']].index
ids_u2 = resumen_pacientes[resumen_pacientes['U2_Limpia']].index
ids_u5 = resumen_pacientes[resumen_pacientes['U5_Nulos_o_Raros']].index

# Creamos los DataFrames finales
df_u2 = df_base[df_base['paciente_id'].isin(ids_u2)].copy()
df_u3 = df_base[df_base['paciente_id'].isin(ids_u3)].copy()
df_u4 = df_base[df_base['paciente_id'].isin(ids_u4)].copy()
df_u5 = df_base[df_base['paciente_id'].isin(ids_u5)].copy()

# --- VERIFICACIÓN DE INTEGRIDAD HONESTA ---
total_filas_original = len(df_base)
total_filas_universos = len(df_u1) + len(df_u2) + len(df_u3) + len(df_u4) + len(df_u5)

print(f"Total de filas en df_base : {total_filas_original}")
print(f"Suma de los Universos     : {total_filas_universos}\n")

if total_filas_original == total_filas_universos and len(df_u5) == 0:
    print("✅ ¡PRUEBA SUPERADA! Los 4 universos cubren el 100% lógicamente.")
elif total_filas_original == total_filas_universos and len(df_u5) > 0:
    print("⚠️ ADVERTENCIA: La suma da bien, pero descubriste el 'Universo 5' (Fechas nulas u otros errores lógicos).")
else:
    print("❌ ALERTA CRÍTICA: La matemática no cierra. Hay pacientes perdidos.")

print("-" * 70)
print(f"U1. Triviales (1 registro): {len(df_u1):>6} filas | {len(ids_u1):>6} pacientes únicos")
print(f"U2. Secuencia Limpia      : {len(df_u2):>6} filas | {len(ids_u2):>6} pacientes únicos")
print(f"U3. Mismo Día             : {len(df_u3):>6} filas | {len(ids_u3):>6} pacientes únicos")
print(f"U4. Solapamientos         : {len(df_u4):>6} filas | {len(ids_u4):>6} pacientes únicos")
if len(df_u5) > 0:
    print(f"U5. Fechas Nulas/Raros    : {len(df_u5):>6} filas | {len(ids_u5):>6} pacientes únicos")
print("="*70 + "\n")


🌌 PARTICIÓN EN MACRO-UNIVERSOS (MECE ESTRICTO)
Total de filas en df_base : 29697
Suma de los Universos     : 29697

⚠️ ADVERTENCIA: La suma da bien, pero descubriste el 'Universo 5' (Fechas nulas u otros errores lógicos).
----------------------------------------------------------------------
U1. Triviales (1 registro):  25188 filas |  25188 pacientes únicos
U2. Secuencia Limpia      :   2657 filas |   1296 pacientes únicos
U3. Mismo Día             :   1531 filas |    660 pacientes únicos
U4. Solapamientos         :     63 filas |     29 pacientes únicos
U5. Fechas Nulas/Raros    :    258 filas |    122 pacientes únicos



### Universo 2: pacientes con exactamente 2 registros

In [4]:
# 1. AISLAMIENTO DEL UNIVERSO 2 (Pacientes con exactamente 2 registros)
# ======================================================================
# Sacamos a los pacientes rotos del U5
df_multi_operable = df_multi[~df_multi['paciente_id'].isin(ids_u5)].copy()

# Contamos registros por paciente
conteo = df_multi_operable['paciente_id'].value_counts()
ids_n2 = conteo[conteo == 2].index

df_n2 = df_multi_operable[df_multi_operable['paciente_id'].isin(ids_n2)].copy()
print(f"Total de pacientes con exactamente 2 registros: {len(ids_n2)}")

Total de pacientes con exactamente 2 registros: 1806


In [5]:
#  2. REPORTE DE IMPACTO ESTRUCTURAL Y TEST DE CORDURA
# ======================================================================
df_univ2 = df_n2.sort_values(['paciente_id', 'fecha_ingreso', 'fecha_egreso']).copy()
df_univ2['numero_fila'] = df_univ2.groupby('paciente_id').cumcount() + 1

# --- TEST DE CORDURA ---
print("="*65)
print("🔍 TEST DE CORDURA: ¿Qué pasa en la Fila 1 vs Fila 2?")
print("="*65)
print(">>> Fila 1 (Inicio):")
print(df_univ2[df_univ2['numero_fila'] == 1]['tipo_egreso'].value_counts(dropna=False).head(3))
print("\n>>> Fila 2 (Fin):")
print(df_univ2[df_univ2['numero_fila'] == 2]['tipo_egreso'].value_counts(dropna=False).head(3))
print("="*65 + "\n")

# --- REPORTE DE IMPACTO ---
total_univ2 = len(df_univ2)

# Regla 8 corregida: Error si tiene flag admin o si la última fila es traslado
mask_traslado_huerfano = (df_univ2['numero_fila'] == 2) & (df_univ2['tipo_egreso'] == 'traslado')
mask_regla_8_corregida = mask_traslado_huerfano | df_univ2['flag_egreso_admin_invalido']

filtros_univ2 = [
    ("1. Sin ID paciente", df_univ2['paciente_id'].isna().sum()), 
    ("2. Sin Hospital", (df_univ2['hospital_origen'].isna() | (df_univ2['hospital_origen'] == '')).sum()),
    ("3. Sin Fecha Ingreso", df_univ2['fecha_ingreso'].isna().sum()),
    ("4. Sin Fecha Egreso", df_univ2['fecha_egreso'].isna().sum()),
    ("5. Fechas Incoherentes (< 0)", df_univ2['flag_fechas_incoherentes'].sum()),
    ("6. Edades Raras", df_univ2['flag_edad_rara'].sum()),
    ("7. Estancias Largas (> 60)", df_univ2['flag_duracion_larga'].sum()),
    ("8. Err. Admin / Traslado Huérfano", mask_regla_8_corregida.sum())
]

print("="*65)
print("📊 REPORTE DE CALIDAD BASE: UNIVERSO 2")
print("="*65)
for nombre, cantidad in filtros_univ2:
    pct = (cantidad / total_univ2) * 100 if total_univ2 > 0 else 0
    print(f"{nombre:<35}: {cantidad:>6} filas | {pct:>6.2f}%")

mask_errores_est_u2 = (df_univ2['paciente_id'].isna() | df_univ2['hospital_origen'].isna() | (df_univ2['hospital_origen'] == '') | df_univ2['fecha_ingreso'].isna() | df_univ2['fecha_egreso'].isna() | df_univ2['flag_fechas_incoherentes'])
univ2_sanos_est = total_univ2 - mask_errores_est_u2.sum()

print("\nRESUMEN DE CALIDAD ESTRUCTURAL (Ignorando 6, 7 y 8):")
print(f"✅ Sanos   : {univ2_sanos_est:>6} filas | {(univ2_sanos_est/total_univ2)*100:>6.2f}%")
print(f"❌ Críticos: {mask_errores_est_u2.sum():>6} filas | {(mask_errores_est_u2.sum()/total_univ2)*100:>6.2f}%")
print("="*65 + "\n")

🔍 TEST DE CORDURA: ¿Qué pasa en la Fila 1 vs Fila 2?
>>> Fila 1 (Inicio):
tipo_egreso
traslado            1739
alta                  36
hospital-externo      14
Name: count, dtype: int64

>>> Fila 2 (Fin):
tipo_egreso
alta                1171
muerte               263
hospital-externo     170
Name: count, dtype: int64

📊 REPORTE DE CALIDAD BASE: UNIVERSO 2
1. Sin ID paciente                 :      0 filas |   0.00%
2. Sin Hospital                    :      0 filas |   0.00%
3. Sin Fecha Ingreso               :      6 filas |   0.17%
4. Sin Fecha Egreso                :     32 filas |   0.89%
5. Fechas Incoherentes (< 0)       :      5 filas |   0.14%
6. Edades Raras                    :      0 filas |   0.00%
7. Estancias Largas (> 60)         :     14 filas |   0.39%
8. Err. Admin / Traslado Huérfano  :    203 filas |   5.62%

RESUMEN DE CALIDAD ESTRUCTURAL (Ignorando 6, 7 y 8):
✅ Sanos   :   3571 filas |  98.86%
❌ Críticos:     41 filas |   1.14%



In [6]:
# ======================================================================
# 3. ANÁLISIS GRANULAR POR BOLSA DE EGRESOS (Solo pacientes sanos)
# ======================================================================
# 1. Aplicamos el filtro básico (Reglas 1 al 5) para separar la paja del trigo
mask_errores = (
    df_n2['paciente_id'].isna() |
    df_n2['hospital_origen'].isna() | (df_n2['hospital_origen'] == '') |
    df_n2['fecha_ingreso'].isna() |
    df_n2['fecha_egreso'].isna() |
    df_n2['flag_fechas_incoherentes']
)
ids_sucios = df_n2[mask_errores]['paciente_id'].unique()
df_n2_limpio = df_n2[~df_n2['paciente_id'].isin(ids_sucios)].copy()

# 2. Definimos la función de la bolsa
def clasificar_bolsa_egresos(serie_egresos):
    egresos = serie_egresos.fillna('nan').astype(str).str.strip().str.lower().tolist()
    cant_traslados = egresos.count('traslado')
    
    if cant_traslados == 0: return "h. Ningún 'traslado' (Ej: 2 altas)"
    elif cant_traslados == 2: return "g. 2 'traslados' (Ping-Pong)"
    elif cant_traslados == 1:
        egresos.remove('traslado')
        otro = egresos[0]
        if otro == 'alta': return "a. 1 traslado + 1 alta"
        elif otro == 'muerte': return "b. 1 traslado + 1 muerte"
        elif otro == 'hospital-externo': return "c. 1 traslado + 1 hosp-externo"
        elif otro == 'hotel': return "d. 1 traslado + 1 hotel"
        elif otro == 'administrativo': return "e. 1 traslado + 1 admin (Error)"
        elif otro == 'desconocido': return "f. 1 traslado + 1 desconocido (Error)"
        elif otro == 'nan': return "❌ 1 traslado + 1 Sin Dato (NaN)"
        else: return f"Otra: 1 traslado + 1 '{otro}'"
    return "No Contemplado"

# 3. Aplicamos la lógica SOLO sobre df_n2_limpio
df_combinaciones = df_n2_limpio.groupby('paciente_id')['tipo_egreso'].apply(clasificar_bolsa_egresos).reset_index(name='categoria')
conteo_egresos = df_combinaciones['categoria'].value_counts()

print("="*65)
print("🎒 DESGLOSE DE COMBINACIONES DE EGRESO (Pacientes Sanos)")
print("="*65)
for cat, cant in conteo_egresos.items():
    print(f"{cat:<40}: {cant:>5} pacientes | {(cant/len(df_combinaciones))*100:>5.2f}%")

# ======================================================================
# 3.5. CHECK DE COHERENCIA DEMOGRÁFICA (Ajustado por cumpleaños)
# ======================================================================
# Agrupamos por paciente para sacar el máximo y mínimo de edad, y valores únicos de sexo
df_demo = df_n2_limpio.groupby('paciente_id').agg(
    edad_max=('edad', 'max'),
    edad_min=('edad', 'min'),
    valores_sexo=('sexo', 'nunique') # Cambiá 'sexo' por la columna real si es necesario
).reset_index()

# Calculamos la diferencia exacta en años
df_demo['dif_edad'] = df_demo['edad_max'] - df_demo['edad_min']

# Aplicamos tus nuevas reglas lógicas:
# Error = salto mayor a 1 año, o cambio de sexo.
pacientes_cambio_edad = (df_demo['dif_edad'] > 1).sum()
pacientes_cambio_sexo = (df_demo['valores_sexo'] > 1).sum()
pacientes_cambio_ambos = ((df_demo['dif_edad'] > 1) & (df_demo['valores_sexo'] > 1)).sum()

total_sanos = len(df_demo)

print("="*70)
print("🧬 CHECK DE COHERENCIA DEMOGRÁFICA (Pacientes Sanos N=2)")
print("="*70)
print(f"Total pacientes analizados: {total_sanos}")
print(f"⚠️ Salto de Edad (> 1 año): {pacientes_cambio_edad:>5} pacientes | {(pacientes_cambio_edad/total_sanos)*100:>5.2f}%")
print(f"⚠️ Cambian de Sexo        : {pacientes_cambio_sexo:>5} pacientes | {(pacientes_cambio_sexo/total_sanos)*100:>5.2f}%")
print(f"🚨 Cambian AMBOS          : {pacientes_cambio_ambos:>5} pacientes | {(pacientes_cambio_ambos/total_sanos)*100:>5.2f}%")
print("="*70)

# ======================================================================
# CONSULTAS RÁPIDAS
# ======================================================================

# 1. Ver detalle de los pacientes que saltaron más de 1 año en edad
# df_n2_limpio[df_n2_limpio['paciente_id'].isin(df_demo[df_demo['dif_edad'] > 1]['paciente_id'])].sort_values(['paciente_id', 'fecha_ingreso'])

# 2. Ver detalle de los pacientes que cambiaron de sexo
#df_n2_limpio[df_n2_limpio['paciente_id'].isin(df_demo[df_demo['valores_sexo'] > 1]['paciente_id'])].sort_values(['paciente_id', 'fecha_ingreso'])

# 3. Ver detalle de los pacientes que cambiaron AMBOS (Edad y Sexo)
# df_n2_limpio[df_n2_limpio['paciente_id'].isin(df_demo[(df_demo['dif_edad'] > 1) & (df_demo['valores_sexo'] > 1)]['paciente_id'])].sort_values(['paciente_id', 'fecha_ingreso'])

🎒 DESGLOSE DE COMBINACIONES DE EGRESO (Pacientes Sanos)
a. 1 traslado + 1 alta                  :  1185 pacientes | 67.02%
b. 1 traslado + 1 muerte                :   268 pacientes | 15.16%
c. 1 traslado + 1 hosp-externo          :   180 pacientes | 10.18%
e. 1 traslado + 1 admin (Error)         :   118 pacientes |  6.67%
h. Ningún 'traslado' (Ej: 2 altas)      :    12 pacientes |  0.68%
d. 1 traslado + 1 hotel                 :     5 pacientes |  0.28%
🧬 CHECK DE COHERENCIA DEMOGRÁFICA (Pacientes Sanos N=2)
Total pacientes analizados: 1768
⚠️ Salto de Edad (> 1 año):    44 pacientes |  2.49%
⚠️ Cambian de Sexo        :    20 pacientes |  1.13%
🚨 Cambian AMBOS          :     9 pacientes |  0.51%


In [7]:
# 4. INSPECCIÓN VISUAL DEL GRUPO 'h' (Sin traslados)
# ======================================================================
ids_grupo_h = df_combinaciones[df_combinaciones['categoria'].str.startswith('h.')]['paciente_id']
df_grupo_h = df_n2[df_n2['paciente_id'].isin(ids_grupo_h)][
    ['paciente_id', 'hospital_origen', 'fecha_ingreso', 'fecha_egreso', 'dias_en_nodo', 'tipo_egreso']
].sort_values(['paciente_id', 'fecha_ingreso'])

print("="*75)
print("🧐 RADIOGRAFÍA DE LOS PACIENTES SIN 'TRASLADO' (Grupo h)")
print("="*75)
print(df_grupo_h.to_string(index=False))

🧐 RADIOGRAFÍA DE LOS PACIENTES SIN 'TRASLADO' (Grupo h)
paciente_id            hospital_origen fecha_ingreso fecha_egreso  dias_en_nodo      tipo_egreso
          2                UPA 17 - QU    2020-04-06   2020-06-23          78.0             alta
          2                   Oñativia    2020-05-29   2020-06-11          13.0            hotel
          3                UPA 17 - QU    2020-04-29   2020-06-12          44.0             alta
          3                   Oñativia    2020-05-25   2020-06-08          14.0             alta
          4                   Oñativia    2020-05-31   2020-06-11          11.0            hotel
          4                UPA 17 - QU    2020-06-08   2020-06-23          15.0             alta
          5                   Oñativia    2020-06-05   2020-06-21          16.0           muerte
          5                UPA 17 - QU    2020-06-08   2020-06-15           7.0 hospital-externo
       1003                   El Cruce    2020-05-02   2020-05-20      

In [8]:
#  5. CLASIFICACIÓN FINAL Y EXPORTACIÓN A EXCEL
# ======================================================================
import os

# 1. Filtro Estructural
mask_errores = (df_n2['paciente_id'].isna() | df_n2['hospital_origen'].isna() | (df_n2['hospital_origen'] == '') | df_n2['fecha_ingreso'].isna() | df_n2['fecha_egreso'].isna() | df_n2['flag_fechas_incoherentes'])
ids_sucios = df_n2[mask_errores]['paciente_id'].unique()
df_n2_limpio = df_n2[~df_n2['paciente_id'].isin(ids_sucios)].copy()

# 2. Clasificación en 3 Sub-Universos
def clasificar_subuniverso(serie_egresos):
    egresos = serie_egresos.fillna('nan').astype(str).str.strip().str.lower().tolist()
    cant_traslados = egresos.count('traslado')
    if cant_traslados == 0: return "Sub_U3_Anomalos_SinTraslado"
    elif cant_traslados == 1:
        egresos.remove('traslado')
        if egresos[0] in ['alta', 'muerte', 'hospital-externo', 'hotel']: return "Sub_U1_CiclosCerradosExitosamente"
        else: return "Sub_U2_CiclosRotos"
    return "Sub_Otros"

clasificacion_final = df_n2_limpio.groupby('paciente_id')['tipo_egreso'].apply(clasificar_subuniverso).reset_index(name='subuniverso')

# 3. Cruzamos la clasificación con los datos completos
df_u2_export = pd.merge(df_n2_limpio, clasificacion_final, on='paciente_id', how='left')

print("="*65)
print("💾 GUARDANDO SUB-UNIVERSOS EN EXCEL...")
print("="*65)

# Creamos una carpetita para ser prolijos (opcional)
os.makedirs('auditoria_u2', exist_ok=True)

# Iteramos sobre cada sub-universo y lo guardamos
for sub_u in df_u2_export['subuniverso'].unique():
    df_temp = df_u2_export[df_u2_export['subuniverso'] == sub_u].sort_values(['paciente_id', 'fecha_ingreso'])
    
    # Nombre del archivo limpio
    nombre_archivo = f"auditoria_u2/{sub_u}.xlsx"
    df_temp.to_excel(nombre_archivo, index=False)
    
    print(f"✅ Guardado: {nombre_archivo} ({len(df_temp['paciente_id'].unique())} pacientes)")

print("="*65 + "\n¡Todo listo y exportado!")

💾 GUARDANDO SUB-UNIVERSOS EN EXCEL...
✅ Guardado: auditoria_u2/Sub_U3_Anomalos_SinTraslado.xlsx (12 pacientes)
✅ Guardado: auditoria_u2/Sub_U1_CiclosCerradosExitosamente.xlsx (1638 pacientes)
✅ Guardado: auditoria_u2/Sub_U2_CiclosRotos.xlsx (118 pacientes)
¡Todo listo y exportado!


<!----
'''

NOTA MENTAL: ESTRATEGIA DE RECONSTRUCCIÓN (Universo 2)
El objetivo de la fase de reconstrucción no es borrar datos, sino "suturar" la historia clínica del paciente basándonos en la verdad que nos reveló su bolsa de egresos.

1. Cuando la bolsa es perfecta pero las fechas mienten (1 Traslado + 1 Alta)
Sabemos empíricamente que el paciente fue derivado y se curó. Si las fechas se solapan (ej: ingresó al Hospital B el martes, pero el Hospital A le dio el egreso recién el jueves), es un típico caso de "Alta Administrativa Tardía".

La Solución: Vamos a usar a la Fila 2 como "ancla de verdad". Si el paciente ingresó físicamente al Hospital B el martes, es imposible que siguiera en el Hospital A. Por lo tanto, vamos a sobreescribir la fecha de egreso del Hospital A para que coincida exactamente con la fecha de ingreso al Hospital B. Sutura temporal perfecta.

2. Cuando la bolsa tiene "2 Altas" (El Grupo H)
Como bien decís, acá no hay un traslado en el medio. Hay dos episodios independientes. Para reconstruir esto, la clave va a ser medir el Gap temporal (los días de diferencia entre el primer egreso y el segundo ingreso).

Si el Gap es largo (ej: > 48 horas): Son claramente dos episodios distintos. El paciente se enfermó, se curó, y a los meses volvió a caer. Solución: No tocamos nada. Simplemente los "desconectamos" y los tratamos como si fueran dos pacientes del Universo 1.

Si el Gap es corto o nulo y los hospitales son distintos: Es un error de carga. Alguien le dio "Alta" en el Hospital A cuando en realidad lo estaban subiendo a la ambulancia. Solución: Le "tachamos" la palabra Alta en la Fila 1 y le forzamos la palabra "Traslado". Así, la historia vuelve a engranar perfectamente.

Si todo es idéntico (mismas fechas, mismo hospital): Es un "dedazo" del data entry. Solución: drop_duplicates y nos quedamos con una sola fila.

3. Cuando el final está arruinado (1 Traslado + 1 Administrativo/Desconocido)
Acá la secuencia de fechas suele estar bien, pero el final del viaje es basura. El paciente llegó al Hospital B y después de un tiempo, en vez de poner si se curó o falleció, el administrativo cerró el caso mal.

La Solución: Como no podemos revivir muertos ni inventar curas, la regla general en salud es aplicar un Cierre Imputado. Vamos a reemplazar ese "administrativo/desconocido" por un valor nuevo (ej: "alta-imputada" o "cierre-forzado") para que el sistema sepa que la internación terminó ahí, pero dejando la marca de agua de que ese dato fue reconstruido por el algoritmo y no por un médico.

'''


->

### Universo 3 (3 registros por paciente)

In [9]:
# 0. AISLAMIENTO DEL UNIVERSO 3 (Pacientes con exactamente 3 registros)
# ======================================================================
# Asumimos que df_multi_operable ya existe (sin los pacientes rotos del U5)
conteo_u3 = df_multi_operable['paciente_id'].value_counts()
ids_n3 = conteo_u3[conteo_u3 == 3].index

df_n3 = df_multi_operable[df_multi_operable['paciente_id'].isin(ids_n3)].copy()
total_univ3_filas = len(df_n3)
total_univ3_pctes = len(ids_n3)


In [10]:
# 1. REPORTE DE IMPACTO ESTRUCTURAL (Sin Regla 8)
# ======================================================================
print("\n" + "="*65)
print("📊 REPORTE DE IMPACTO: UNIVERSO 3 (Pacientes con 3 registros)")
print("="*65)
print(f"Total de episodios en el universo 3: {total_univ3_filas} filas ({total_univ3_pctes} pacientes)\n")

filtros_univ3 = [
    ("1. Sin ID paciente", df_n3['paciente_id'].isna().sum()), 
    ("2. Sin Hospital", (df_n3['hospital_origen'].isna() | (df_n3['hospital_origen'] == '')).sum()),
    ("3. Sin Fecha Ingreso", df_n3['fecha_ingreso'].isna().sum()),
    ("4. Sin Fecha Egreso", df_n3['fecha_egreso'].isna().sum()),
    ("5. Fechas Incoherentes (< 0)", df_n3['flag_fechas_incoherentes'].sum()),
    ("6. Edades Raras", df_n3['flag_edad_rara'].sum()),
    ("7. Estancias Largas (> 60)", df_n3['flag_duracion_larga'].sum())
]

for nombre, cantidad in filtros_univ3:
    pct = (cantidad / total_univ3_filas) * 100 if total_univ3_filas > 0 else 0
    print(f"{nombre:<35}: {cantidad:>6} filas | {pct:>6.2f}%")

# Filtro estructural duro (Reglas 1 a 5)
mask_errores_est_u3 = (
    df_n3['paciente_id'].isna() | df_n3['hospital_origen'].isna() | 
    (df_n3['hospital_origen'] == '') | df_n3['fecha_ingreso'].isna() | 
    df_n3['fecha_egreso'].isna() | df_n3['flag_fechas_incoherentes']
)

filas_sanas_est = total_univ3_filas - mask_errores_est_u3.sum()

print("\nRESUMEN DE CALIDAD - ESTRUCTURAL (Ignorando 6 y 7):")
print(f"✅ Casos Estructuralmente Sanos : {filas_sanas_est:>6} filas | {(filas_sanas_est/total_univ3_filas)*100:>6.2f}%")
print(f"❌ Casos con Errores Críticos   : {mask_errores_est_u3.sum():>6} filas | {(mask_errores_est_u3.sum()/total_univ3_filas)*100:>6.2f}%")
print("="*65 + "\n")




📊 REPORTE DE IMPACTO: UNIVERSO 3 (Pacientes con 3 registros)
Total de episodios en el universo 3: 321 filas (107 pacientes)

1. Sin ID paciente                 :      0 filas |   0.00%
2. Sin Hospital                    :      0 filas |   0.00%
3. Sin Fecha Ingreso               :      1 filas |   0.31%
4. Sin Fecha Egreso                :      8 filas |   2.49%
5. Fechas Incoherentes (< 0)       :      0 filas |   0.00%
6. Edades Raras                    :      0 filas |   0.00%
7. Estancias Largas (> 60)         :      3 filas |   0.93%

RESUMEN DE CALIDAD - ESTRUCTURAL (Ignorando 6 y 7):
✅ Casos Estructuralmente Sanos :    312 filas |  97.20%
❌ Casos con Errores Críticos   :      9 filas |   2.80%



In [11]:
# 2. LIMPIEZA EN CASCADA Y FILTRO DE PACIENTES SANOS
# ======================================================================
# Sacamos a los pacientes que tienen AL MENOS UNA fila rota
ids_sucios_u3 = df_n3[mask_errores_est_u3]['paciente_id'].unique()
df_n3_limpio = df_n3[~df_n3['paciente_id'].isin(ids_sucios_u3)].copy()

# Detectamos duplicados exactos (mismo paciente, hospital, fechas y egreso)
# Contamos cuántas filas ÚNICAS le quedan a cada paciente
conteo_filas_unicas = df_n3_limpio.drop_duplicates(
    subset=['paciente_id', 'hospital_origen', 'fecha_ingreso', 'fecha_egreso', 'tipo_egreso']
).groupby('paciente_id').size()

# Clasificamos según la cascada
ids_falsos_n3 = conteo_filas_unicas[conteo_filas_unicas < 3].index
ids_reales_n3 = conteo_filas_unicas[conteo_filas_unicas == 3].index

df_n3_reales = df_n3_limpio[df_n3_limpio['paciente_id'].isin(ids_reales_n3)].copy()

print("="*75)
print("🌊 EFECTO CASCADA: DETECCIÓN DE FALSOS N=3")
print("="*75)
print(f"Pacientes sanos evaluados: {len(df_n3_limpio['paciente_id'].unique())}")
print(f"👇 Caen al U1/U2 (Tienen filas duplicadas exactas): {len(ids_falsos_n3)} pacientes")
print(f"🔥 Sobreviven como N=3 Reales para auditar       : {len(ids_reales_n3)} pacientes")
print("="*75 + "\n")



🌊 EFECTO CASCADA: DETECCIÓN DE FALSOS N=3
Pacientes sanos evaluados: 99
👇 Caen al U1/U2 (Tienen filas duplicadas exactas): 0 pacientes
🔥 Sobreviven como N=3 Reales para auditar       : 99 pacientes



In [12]:
# 3. LÓGICA DE LA BOLSA DE EGRESOS (Solo N=3 Reales)
# ======================================================================
def clasificar_bolsa_n3(serie_egresos):
    egresos = serie_egresos.fillna('nan').astype(str).str.strip().str.lower().tolist()
    cant_traslados = egresos.count('traslado')
    cant_muertes = egresos.count('muerte')
    
    # 🟡 SUB-UNIVERSO 3: Anomalías Temporales/Zombies
    if cant_muertes >= 2:
        return "🟡 SUB-U3: Zombie Multiverso (2+ muertes)"
        
    # 🔴 SUB-UNIVERSO 2: Historias Incompletas o Rotas
    if 'administrativo' in egresos or 'desconocido' in egresos or 'nan' in egresos:
        return "🔴 SUB-U2: Cierres Basura (Contiene Admin/Desc/NaN)"
    if cant_traslados == 3:
        return "🔴 SUB-U2: El Agujero Negro (3 Traslados)"
        
    # 🟢 SUB-UNIVERSO 1: Triangulaciones y Ciclos Válidos
    if cant_traslados == 2:
        return "🟢 SUB-U1: Derivación en Cadena (2 Traslados + 1 Cierre)"
    elif cant_traslados == 1:
        return "🟢 SUB-U1: Recaída Derivada (1 Traslado + 2 Cierres)"
    elif cant_traslados == 0:
        return "🟢 SUB-U1: Crónico Puro (0 Traslados, 3 Cierres)"
        
    return "⚫ NO CLASIFICADO"

# Aplicamos la lógica de la bolsa
df_combinaciones_u3 = df_n3_reales.groupby('paciente_id')['tipo_egreso'].apply(clasificar_bolsa_n3).reset_index(name='subuniverso')
conteo_egresos_u3 = df_combinaciones_u3['subuniverso'].value_counts()

print("="*75)
print("🎒 AUDITORÍA DE SUB-UNIVERSOS (Pacientes N=3 Reales)")
print("="*75)
for subuniv, cantidad in conteo_egresos_u3.items():
    pct = (cantidad / len(df_combinaciones_u3)) * 100
    print(f"{subuniv:<55}: {cantidad:>5} casos | {pct:>5.2f}%")
print("="*75)

🎒 AUDITORÍA DE SUB-UNIVERSOS (Pacientes N=3 Reales)
🟢 SUB-U1: Derivación en Cadena (2 Traslados + 1 Cierre):    88 casos | 88.89%
🔴 SUB-U2: Cierres Basura (Contiene Admin/Desc/NaN)     :    10 casos | 10.10%
🟢 SUB-U1: Crónico Puro (0 Traslados, 3 Cierres)        :     1 casos |  1.01%


In [ ]:
#  4. CONSULTAS RÁPIDAS POR SUB-UNIVERSO (N=3)
# ======================================================================
# Primero cruzamos la clasificación con los datos completos de los N=3 Reales
# (Descomentá la línea de abajo para poder hacer las consultas)
# df_u3_clasificado = pd.merge(df_n3_reales, df_combinaciones_u3, on='paciente_id', how='left')

# ----------------------------------------------------------------------
# 🟢 SUB-UNIVERSO 1: CASOS VÁLIDOS (Descomentá el que quieras ver)
# ----------------------------------------------------------------------
# 1. Derivación en Cadena (Pasó por 2 hospitales y quedó internado/de alta en el 3ro)
# df_u3_clasificado[df_u3_clasificado['subuniverso'].str.contains('Derivación en Cadena')].sort_values(['paciente_id', 'fecha_ingreso'])

# 2. Recaída Derivada (Se curó una vez, recayó meses después, entró por guardia y lo internaron)
# df_u3_clasificado[df_u3_clasificado['subuniverso'].str.contains('Recaída Derivada')].sort_values(['paciente_id', 'fecha_ingreso'])

# 3. Crónico Puro (3 internaciones completamente separadas en el tiempo)
# df_u3_clasificado[df_u3_clasificado['subuniverso'].str.contains('Crónico Puro')].sort_values(['paciente_id', 'fecha_ingreso'])

# ----------------------------------------------------------------------
# 🔴 SUB-UNIVERSO 2: HISTORIAS ROTAS
# ----------------------------------------------------------------------
# 4. Cierres Basura (El viaje venía bien pero el último hospital cargó cualquier cosa)
# df_u3_clasificado[df_u3_clasificado['subuniverso'].str.contains('Cierres Basura')].sort_values(['paciente_id', 'fecha_ingreso'])

# 5. El Agujero Negro (Lo derivaron 3 veces y nunca nadie le dio el alta o lo dio por fallecido)
# df_u3_clasificado[df_u3_clasificado['subuniverso'].str.contains('Agujero Negro')].sort_values(['paciente_id', 'fecha_ingreso'])

# ----------------------------------------------------------------------
# 🟡 SUB-UNIVERSO 3: ANOMALÍAS GRAVES
# ----------------------------------------------------------------------
# 6. Zombie Multiverso (El paciente registra 2 o más muertes)
# df_u3_clasificado[df_u3_clasificado['subuniverso'].str.contains('Zombie Multiverso')].sort_values(['paciente_id', 'fecha_ingreso'])

# ======================================================================
# 💾 EXPORTACIÓN A EXCEL (Opcional)
# ======================================================================
# Si en algún momento los mentores te piden la base de N=3 partida por caso,
# descomentá este bloque y te guarda todos los excels automáticos:

# import os
# os.makedirs('auditoria_u3', exist_ok=True)
# for sub_u in df_u3_clasificado['subuniverso'].dropna().unique():
#     df_temp = df_u3_clasificado[df_u3_clasificado['subuniverso'] == sub_u].sort_values(['paciente_id', 'fecha_ingreso'])
#     nombre_limpio = sub_u.replace(':', '').replace('/', '-').replace('\\', '-').replace(' ', '_')
#     df_temp.to_excel(f"auditoria_u3/{nombre_limpio}.xlsx", index=False)
#     print(f"✅ Guardado: {nombre_limpio}.xlsx")

In [ ]:
print("--- (Percentiles) ---")

# Calculamos el percentil 90
p90 = df['dias_en_nodo'].quantile(0.90)
print(f"El 90% de los pacientes en la base general tiene {p90} días o menos de internación.\n")

# Tabla de frecuencias con porcentaje acumulado
dist_dias = df['dias_en_nodo'].value_counts(normalize=True).sort_index().reset_index()
dist_dias.columns = ['dias_en_nodo', 'porcentaje']
dist_dias['porcentaje'] = dist_dias['porcentaje'] * 100
dist_dias['porcentaje_acumulado'] = dist_dias['porcentaje'].cumsum()

print("Distribución de días (mostrando hasta pasar el 90% aprox):")
# Filtramos para mostrar hasta que el acumulado llegue un poquito más del 90%
display(dist_dias[dist_dias['porcentaje_acumulado'] <= 95])